In [ ]:
%pip install Faker

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = "raw_data"

assert catalog, "catalog widget must be set"
assert schema, "schema widget must be set"

volume_folder = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"Generating CDC data into {volume_folder}")

In [ ]:
from pyspark.sql import functions as F
from faker import Faker
from collections import OrderedDict
import uuid
import random

fake = Faker()

fake_firstname = F.udf(fake.first_name)
fake_lastname = F.udf(fake.last_name)
fake_email = F.udf(fake.ascii_company_email)
fake_date = F.udf(lambda: fake.date_time_this_month().strftime("%m-%d-%Y %H:%M:%S"))
fake_address = F.udf(fake.address)
operations = OrderedDict([("APPEND", 0.5), ("DELETE", 0.1), ("UPDATE", 0.3), (None, 0.01)])
fake_operation = F.udf(lambda: fake.random_elements(elements=operations, length=1)[0])
fake_id = F.udf(lambda: str(uuid.uuid4()) if random.uniform(0, 1) < 0.98 else None)

# Generate customers CDC data
df = spark.range(0, 100000).repartition(100)
df = df.withColumn("id", fake_id())
df = df.withColumn("firstname", fake_firstname())
df = df.withColumn("lastname", fake_lastname())
df = df.withColumn("email", fake_email())
df = df.withColumn("address", fake_address())
df = df.withColumn("operation", fake_operation())
df_customers = df.withColumn("operation_date", fake_date())
df_customers.repartition(100).write.format("json").mode("overwrite").save(volume_folder + "/customers")
print(f"Wrote {df_customers.count()} customer CDC records")

# Generate transactions CDC data
df = spark.range(0, 10000).repartition(20)
df = df.withColumn("id", fake_id())
df = df.withColumn("transaction_date", fake_date())
df = df.withColumn("amount", F.round(F.rand() * 1000))
df = df.withColumn("item_count", F.round(F.rand() * 10))
df = df.withColumn("operation", fake_operation())
df = df.withColumn("operation_date", fake_date())
df = df.withColumn("t_id", F.monotonically_increasing_id()).join(
    spark.read.json(volume_folder + "/customers")
    .select("id")
    .withColumnRenamed("id", "customer_id")
    .withColumn("t_id", F.monotonically_increasing_id()),
    "t_id",
).drop("t_id")
df.repartition(10).write.format("json").mode("overwrite").save(volume_folder + "/transactions")
print(f"Wrote {df.count()} transaction CDC records")